# Exercise 3 - indices

## Imports

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import transform
from rasterio.windows import from_bounds
import spectral.io.envi as envi
from pystac_client import Client
import planetary_computer
from datetime import datetime

ACQUISITION_DATE = '2025-06-17'

BASE_DIR = Path().cwd() / "data" / "you-shall-not-pass" / "Obrazy lotnicze"
hdr_path = BASE_DIR / '221000_Odra_HS_Blok_A_008_VS_join_atm.hdr'

print(f"Target Date: {ACQUISITION_DATE}")
print(f"Airborne Header Path: {hdr_path}")

## Load Airborne Data Cube

Only a subset is loaded to save memory ;D

In [ ]:
if not hdr_path.exists():
    raise FileNotFoundError(f"Cannot find airborne data at {hdr_path}")

img = envi.open(hdr_path)
meta = img.metadata
wavelengths = np.array([float(x) for x in meta['wavelength']])

# Read a subset (1000x1000) from the center to keep memory usage low
cy, cx = img.nrows // 2, img.ncols // 2
cube = img.read_subregion((cy-500, cy+500), (cx-500, cx+500))

# Extract geographic bounds of the subset for Sentinel-2 alignment
with rasterio.open(hdr_path) as src:
    # Subset window in pixel coordinates
    air_win = rasterio.windows.Window(cx-500, cy-500, 1000, 1000)
    # Get bounds in airborne CRS (EPSG:2177)
    air_left, air_bottom, air_right, air_top = src.window_bounds(air_win)
    # Transform corners to WGS84 (lon/lat) for STAC and Sentinel-2 search
    lons, lats = transform(src.crs, 'EPSG:4326', [air_left, air_right], [air_bottom, air_top])
    subset_bbox = [lons[0], lats[0], lons[1], lats[1]]
    center_lon, center_lat = np.mean(lons), np.mean(lats)

print(f"Loaded subset shape: {cube.shape}")
print(f"Subset center (lon, lat): {center_lon:.4f}, {center_lat:.4f}")

## Display False-Color Composite

Appropriate bands for R, G and B are selected to represent RGB visualization for the airborn data

In [ ]:
r_idx = np.argmin(np.abs(wavelengths - 670))
g_idx = np.argmin(np.abs(wavelengths - 560))
b_idx = np.argmin(np.abs(wavelengths - 490))

rgb_air = cube[:, :, [r_idx, g_idx, b_idx]]
# Simple stretch for visualization
rgb_air = (rgb_air - np.percentile(rgb_air, 2)) / (np.percentile(rgb_air, 98) - np.percentile(rgb_air, 2))
rgb_air = np.clip(rgb_air, 0, 1)

plt.figure(figsize=(8, 8))
plt.imshow(rgb_air)
plt.title("Airborne False Color RGB")
plt.axis('off')
plt.show()

## Description of Methods

Below are the primary methods used in this notebook:

1. **Hyperspectral Data Loading**: We use the `spectral` (SPy) library to open ENVI datasets. This allows us to access metadata like wavelengths and efficiently read sub-regions of large data cubes.
2. **Spatial Coordinate Transformation**: Using `rasterio.warp.transform`, we convert the local coordinate system of the airborne data (Poland CS2000) into WGS84 (lon/lat). This is essential for matching airborne data with global satellite imagery.
3. **Spectral Band Matching**: To find specific wavelengths (like 705nm for Chl-a), we calculate the absolute difference between the sensor's band centers and our target, then select the band with the minimum distance (`np.argmin`).
4. **Spatiotemporal Data Discovery (STAC)**: We query the SpatioTemporal Asset Catalog (STAC) via the Microsoft Planetary Computer API. This allows us to filter Sentinel-2 scenes by cloud cover, geographic bounding box, and acquisition date.
5. **Windowed Reading**: We use `rasterio.windows.from_bounds` to read only the specific area of the Sentinel-2 image that overlaps with our airborne flight line. This avoids downloading entire satellite tiles (which are ~1GB each).
6. **Spectral Indices calculation**: We use normalized ratios of reflectance values. Water indices often rely on the distinct slope of the reflectance curve in the Red and Near-Infrared (NIR) regions to estimate chemical concentrations.

## Calculate Water Quality Indices for the airborn data

Formulas used:
- **Chl-a**: Ratio of 705nm / 665nm (Red edge / Red)
- **DOC**: Ratio of 560nm / 665nm (Green / Red)
- **Turbidity**: Reflected intensity at 700nm.

### Explanation of the index exraction method:

Since the bands might not be precisely at a given wavelength, the value is interpolated from the nearest bands. It removes wanted wavelength from the spectrum and then selects position where the difference is minimal (it's absolute at this point).

In [ ]:
def safe_ratio(n, d, eps=1e-6):
    return n / (d + eps)

idx_705 = np.argmin(np.abs(wavelengths - 705))
idx_665 = np.argmin(np.abs(wavelengths - 665))
idx_560 = np.argmin(np.abs(wavelengths - 560))
idx_700 = np.argmin(np.abs(wavelengths - 700))

chla_air = safe_ratio(cube[:, :, idx_705], cube[:, :, idx_665])
doc_air = safe_ratio(cube[:, :, idx_560], cube[:, :, idx_665])
turb_air = cube[:, :, idx_700]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (data, name) in enumerate(zip([chla_air, doc_air, turb_air], ['Chl-a', 'DOC', 'Turbidity'])):
    im = axes[i].imshow(data, cmap='viridis')
    axes[i].set_title(f"Airborne {name}")
    plt.colorbar(im, ax=axes[i])
plt.show()

In [ ]:
import certifi
import os

os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

## Download Sentinel-2 Data for a given acquisition date


In [ ]:
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)

# Search using the bounding box of the airborne flight line subset
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=subset_bbox,
    datetime=ACQUISITION_DATE,
    query={"eo:cloud_cover": {"lt": 10}} # Lower cloud cover threshold for clearer data
)

items = list(search.items())
if not items:
    print("No clear Sentinel-2 scenes found for the specific date. Checking nearest 15 days...")
    search = catalog.search(
        collections=["sentinel-2-l2a"], 
        bbox=subset_bbox, 
        datetime="2025-06-10/2025-06-25", 
        query={"eo:cloud_cover": {"lt": 10}},
        sortby="datetime"
    )
    items = list(search.items())

if not items:
    raise ValueError("No suitable Sentinel-2 imagery found. Consider increasing cloud threshold or date range.")

best_item = items[0]
print(f"Selected Sentinel-2 Scene: {best_item.id} ({best_item.datetime})")

# URLs of bands that will be analysed
assets = best_item.assets
bands_s2 = {
    'B03': assets['B03'].href, # Green
    'B04': assets['B04'].href, # Red
    'B05': assets['B05'].href, # Red Edge 1 (~705nm)
    'B11': assets['B11'].href  # SWIR 1 (turbidity proxy)
}

## Calculate Water Quality Indices for Sentinel-2 data



In [ ]:
data_s2 = {}
with rasterio.open(bands_s2['B04']) as src:
    # Calculate the exact pixel window in Sentinel-2 that matches the airborne bounds
    s2_win = from_bounds(subset_bbox[0], subset_bbox[1], subset_bbox[2], subset_bbox[3], src.transform)
    # Ensure window is within image bounds
    s2_win = s2_win.intersection(rasterio.windows.Window(0, 0, src.width, src.height))
    
    data_s2['B04'] = src.read(1, window=s2_win).astype(float) / 10000.0

with rasterio.open(bands_s2['B03']) as src:
    data_s2['B03'] = src.read(1, window=s2_win).astype(float) / 10000.0

with rasterio.open(bands_s2['B05']) as src:
    # Read the same window, resampling to match B04 (10m)
    data_s2['B05'] = src.read(1, window=s2_win, out_shape=data_s2['B04'].shape, resampling=rasterio.enums.Resampling.bilinear).astype(float) / 10000.0

with rasterio.open(bands_s2['B11']) as src:
    data_s2['B11'] = src.read(1, window=s2_win, out_shape=data_s2['B04'].shape, resampling=rasterio.enums.Resampling.bilinear).astype(float) / 10000.0

chla_s2 = safe_ratio(data_s2['B05'], data_s2['B04'])
doc_s2 = safe_ratio(data_s2['B03'], data_s2['B04'])
turb_s2 = data_s2['B11']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(chla_s2, cmap='viridis', vmin=0, vmax=2)
axes[0].set_title("Sentinel-2 Chl-a Proxy")
axes[1].imshow(doc_s2, cmap='viridis', vmin=0, vmax=2)
axes[1].set_title("Sentinel-2 DOC Proxy")
axes[2].imshow(turb_s2, cmap='viridis')
axes[2].set_title("Sentinel-2 Turbidity Proxy")
plt.show()

## Compare Results (values)

Mean of the values is compared, the difference is significant, as the result of the resolution difference (1m vs 10m).

In [ ]:
print(f"Airborne Chl-a: {np.nanmean(chla_air):.4f}")
print(f"Sentinel-2 Chl-a: {np.nanmean(chla_s2):.4f}")
print(f"Relative Difference (Chl-a): {abs(np.nanmean(chla_air)-np.nanmean(chla_s2))/np.nanmean(chla_air)*100:.2f}%")

print(f"\nAirborne DOC: {np.nanmean(doc_air):.4f}")
print(f"Sentinel-2 DOC: {np.nanmean(doc_s2):.4f}")
print(f"Relative Difference (DOC): {abs(np.nanmean(doc_air)-np.nanmean(doc_s2))/np.nanmean(doc_air)*100:.2f}%")

print(f"\nAirborne Turbidity: {np.nanmean(turb_air):.4f}")
print(f"Sentinel-2 Turbidity: {np.nanmean(turb_s2):.4f}")
print(f"Relative Difference (Turbidity): {abs(np.nanmean(turb_air)-np.nanmean(turb_s2))/np.nanmean(turb_air)*100:.2f}%")

## Visual Comparison Side-by-Side

We plot the indices from both sensors side-by-side to visually assess the spatial distribution and patterns. Note the resolution and footprint differences.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 18))

# Chl-a
im0 = axes[0, 0].imshow(chla_air, cmap='viridis', vmin=0, vmax=2)
axes[0, 0].set_title("Airborne Chl-a")
plt.colorbar(im0, ax=axes[0, 0])
im1 = axes[0, 1].imshow(chla_s2, cmap='viridis', vmin=0, vmax=2)
axes[0, 1].set_title("Sentinel-2 Chl-a")
plt.colorbar(im1, ax=axes[0, 1])

# DOC
im2 = axes[1, 0].imshow(doc_air, cmap='viridis', vmin=0, vmax=2)
axes[1, 0].set_title("Airborne DOC")
plt.colorbar(im2, ax=axes[1, 0])
im3 = axes[1, 1].imshow(doc_s2, cmap='viridis', vmin=0, vmax=2)
axes[1, 1].set_title("Sentinel-2 DOC")
plt.colorbar(im3, ax=axes[1, 1])

# Turbidity
im4 = axes[2, 0].imshow(turb_air, cmap='viridis')
axes[2, 0].set_title("Airborne Turbidity")
plt.colorbar(im4, ax=axes[2, 0])
im5 = axes[2, 1].imshow(turb_s2, cmap='viridis')
axes[2, 1].set_title("Sentinel-2 Turbidity")
plt.colorbar(im5, ax=axes[2, 1])

plt.tight_layout()
plt.show()